## このノートの位置づけ

初めて読む方は [CLNとは何か](00_what_is_cln.ipynb) → [展開点とエネルギー](01_expansion_and_energy.ipynb) の順にご覧ください。

ここでは **Type1の4要素 L-1・R0・L1・R2** について、A法とT法の式が異なっても、復元した場から同じ回路定数が得られることを詳しく検証します。丸線・正の実数展開点の範囲です。既存の境界条件や規格化を修正して一致させたという話ではありません。

下の記号計算セルにはWolfram実行環境が必要です。保存済みの結果はGitHub上で読むことができます。有限段の回路は局所近似であり、全周波数の厳密解ではありません。


# Type1: the same circuit from A and T

This is a FEM-free derivation followed by finite-element reproduction for a homogeneous circular wire, real positive expansion point, and **four Type1 elements** $L_{-1},R_0,L_1,R_2$. It is not a proof for every stage, complex shifts, or general 3D A-Phi/T-Omega/A-T spaces.

## Initial normalization and the common correction

Use $x=r/a$, $z=\mu\sigma a^2s$, and normalized unit-voltage fields
$$e=\frac{I_0(\sqrt z x)}{I_0(\sqrt z)},\qquad h=\frac{I_1(\sqrt z x)}{\sqrt z I_0(\sqrt z)},\qquad Y=2\pi h(1).$$
Define $D(E,H)=2\pi\int_0^1x(E^2+zH^2)dx$ and $P(E)=2\pi\int_0^1xE^2dx$.
The initial unit-current fields are $(e,h)/Y$, so
$$L_{-1}=\frac{1}{zY},\qquad E_0=\frac{e/Y}{zL_{-1}}=e,\qquad R_0=1/P(e).$$
Writing $W=-Y'$, the power identity gives $P(e)=Y-zW=Y+zY'$.

The next common physical fields are
$$E_1=R_0z\partial_z e,\qquad H_1=R_0(h+z\partial_z h).$$
They satisfy
$$\frac{(xH_1)'}x=E_1+R_0e,\qquad E_1'=zH_1,\qquad E_1(1)=0.$$
The impressed term $R_0e$ must remain in Ampere's law.

For **A**, solve $-\Delta u+zu=R_0e$ with $u(1)=0$, and reconstruct $E_1=-zu$, $H_1=-u'$. For **T**, solve for $t=H_1$ and reconstruct $E_1=(xt)'/x-R_0e$. Its natural boundary condition is $E_1(1)=0$. The WLS checks both equations, both reconstructions and the integration-by-parts boundary terms.

The difference-field energy identity $[x\delta E\delta H]_0^1=\int_0^1x((\delta E)^2+z(\delta H)^2)dx$ establishes uniqueness for real $z>0$, regular axis and these boundary data. Thus both formulations select the same physical fields, and hence the same energy-derived elements:
$$L_1=D(E_1,H_1)/z,\quad E_2=e+E_1/(zL_1),\quad R_2=1/P(E_2).$$

## Independent circuit extraction

The same elements are extracted independently by continued-fraction division of $G(z)=z/Z(z)=zY(z)$, then inversion of each coefficient. With $q=z-z_0$,
$$Z_{\rm CLN}(z)=z\left[\frac1{L_{-1}}+\frac1{R_0/q+\left(1/L_1+q/R_2\right)^{-1}}\right]^{-1}.$$
The four-element ladder matches through cubic order near the expansion point, not globally. Type1 elements are not Type2 elements. Resistance unit: $1/(\sigma a^2)$; inductance unit: $\mu$.

## Verification scope

The Wolfram script checks 13 exact identities and four high-precision numerical energy/Cauer comparisons. The independent NGSolve script uses two weighted meridian formulations: an axial A potential, and $H_\theta=xv$ for T. It checks field profiles, free-DOF linear residuals, positive elements and mesh convergence. These reduced symmetry formulations do not validate full 3D mixed spaces.

Codex executed the FE script locally and separately in the active radia-ngsolve Python runtime (NGSolve 6.2.2607). This is two-runtime reproduction of the same script; the independent mathematical routes are A, T, and the Bessel oracle. The radia-ngsolve MCP status was checked; execution used its reported Python environment, not an MCP arbitrary-script runner. No independent Opus review is claimed for this Type1 addition.

Install Python packages `ngsolve`, `mpmath`, `numpy` and a Wolfram runtime. Put `wolframscript` on PATH or set `WOLFRAMSCRIPT`.


In [1]:
from pathlib import Path
import os, shutil, subprocess, tempfile, json, sys
root = Path.cwd().parent if Path.cwd().name == "docs" else Path.cwd()
exe = os.environ.get("WOLFRAMSCRIPT") or shutil.which("wolframscript")
if not exe:
    raise RuntimeError("Install wolframscript or set WOLFRAMSCRIPT")
with tempfile.TemporaryDirectory() as tmp:
    output = Path(tmp)/"type1.json"
    run = subprocess.run([exe,"-file",str(root/"mathematica/derive_type1_same_circuit.wls"),str(output)],capture_output=True,text=True)
    print("\n".join(line for line in run.stdout.splitlines() if not line.startswith("RESULT_FILE ")))
    if run.returncode:
        raise RuntimeError(run.stderr or "Wolfram checks failed")
    symbolic = json.loads(output.read_text())
    assert all(symbolic["symbolicChecks"].values())
    assert all(c["passed"] for c in symbolic["cases"])


PASS initial_Ampere
PASS initial_Faraday
PASS initial_power_identity
PASS E0_normalization
PASS A_correction_equation
PASS A_reconstructed_H1
PASS T_correction_equation
PASS T_reconstructed_E1
PASS common_Faraday
PASS A_essential_u_zero
PASS T_natural_E_zero
PASS A_boundary_identity
PASS T_boundary_identity
PASS Type1_energy_vs_Cauer z=1/10 relative_error=0``60.
PASS Type1_energy_vs_Cauer z=1 relative_error=0``60.
PASS Type1_energy_vs_Cauer z=8*Pi^2 relative_error=0``60.
PASS Type1_energy_vs_Cauer z=100 relative_error=0``60.


In [2]:
with tempfile.TemporaryDirectory() as tmp:
    output = Path(tmp)/"fe.json"
    run = subprocess.run([sys.executable,str(root/"validation/type1_ngsolve.py"),"--output",str(output)],capture_output=True,text=True)
    if run.returncode:
        raise RuntimeError(run.stderr or run.stdout)
    print(run.stdout)
    fe = json.loads(output.read_text())
assert fe["passed"]
# Cross-check the NGSolve script's independent mpmath oracle with Wolfram.
wl_values = [float(v.split("`")[0].replace("*^","e")) for v in symbolic["cases"][2]["portElements"]]
assert max(abs(a/b-1) for a,b in zip(fe["oracle"],wl_values)) < 1e-12
print("PASS Wolfram / mpmath analytic oracle agreement < 1e-12")
for name in ("A","T"):
    print(name, "fine-mesh E1/H1 relative RMS:", fe["cases"][-1][name]["E1_H1_profile_relative_RMS"])


h=0.3: A error=3.758e-02, T error=2.040e-02, residual=6.380e-16
h=0.15: A error=2.097e-03, T error=1.111e-03, residual=4.245e-16
h=0.075: A error=2.671e-05, T error=1.400e-05, residual=1.405e-15
h=0.05: A error=2.096e-06, T error=1.104e-06, residual=2.767e-15
PASS A/T fine-mesh relative difference 3.200e-06

PASS Wolfram / mpmath analytic oracle agreement < 1e-12
A fine-mesh E1/H1 relative RMS: [3.388771308963587e-05, 0.00026089967049477015]
T fine-mesh E1/H1 relative RMS: [0.00011497764707454536, 3.286900079821e-05]


In [3]:
# Recorded second-runtime execution, retained as public numerical evidence.
radia = json.loads((root/"docs/data/type1_radia_ngsolve.json").read_text())
assert radia["passed"]
error = max(abs(a/b-1) for name in ("A","T") for a,b in zip(fe["cases"][-1][name]["elements"],radia["cases"][-1][name]["elements"]))
assert error < 1e-10
print("Recorded radia-ngsolve runtime version:",radia["ngsolve_version"])
print("PASS local / recorded radia-ngsolve element difference:",error)


Recorded radia-ngsolve runtime version: 6.2.2607
PASS local / recorded radia-ngsolve element difference: 5.551115123125783e-16
